# Automate a Rigol Oscilloscope with Python (PyVISA + SCPI)

This notebook follows the steps described in the TestFlow blog post:
[How to Automate a Rigol Oscilloscope with Python](https://testflowinc.com/blog/automate-rigol-oscilloscope-python-scpi-pyvisa-guide)

It demonstrates the basic PyVISA/SCPI workflow:
1. List VISA resources
2. Connect to the scope
3. Query identity
4. Run automated measurements
5. Capture waveform data
6. Convert to voltage and plot

In [ ]:
import pyvisa
import numpy as np
import matplotlib.pyplot as plt

# For LAN use TCPIP0::192.168.178.70::INSTR
# For USB use the string from rm.list_resources()
VISA_RESOURCE = "TCPIP0::192.168.178.70::INSTR"
TIMEOUT_MS = 10_000
CHANNEL = 1

## Step 1: Find and connect to the scope

In [ ]:
rm = pyvisa.ResourceManager('@py')
print(rm.list_resources())

scope = rm.open_resource(VISA_RESOURCE, timeout=TIMEOUT_MS)
print(scope.query('*IDN?').strip())

## Step 2: Automated measurements

In [ ]:
def measure(scope, item, channel='CHANnel1'):
    return float(scope.query(f':MEASure:ITEM? {item},{channel}'))

scope.write(':AUToscale')
import time
time.sleep(1.0)

vpp = measure(scope, 'VPP', f'CHANnel{CHANNEL}')
freq = measure(scope, 'FREQuency', f'CHANnel{CHANNEL}')
vavg = measure(scope, 'VAVG', f'CHANnel{CHANNEL}')

INVALID = 9.9e37
print(f'Vpp  = {vpp:.3f} V' if vpp < INVALID else 'Vpp  = invalid')
print(f'f    = {freq:.1f} Hz' if freq < INVALID else 'f    = invalid')
print(f'Vavg = {vavg:.3f} V' if vavg < INVALID else 'Vavg = invalid')

## Step 3: Capture the waveform itself

In [ ]:
scope.write(f':WAVeform:SOURce CHANnel{CHANNEL}')
scope.write(':WAVeform:MODE NORMal')
scope.write(':WAVeform:FORMat BYTE')

xinc = float(scope.query(':WAVeform:XINCrement?'))
yinc = float(scope.query(':WAVeform:YINCrement?'))
yorig = float(scope.query(':WAVeform:YORigin?'))
yref = float(scope.query(':WAVeform:YREFerence?'))

raw = scope.query_binary_values(':WAVeform:DATA?', datatype='B', container=np.array)
volts = (raw - yorig - yref) * yinc
time_s = np.arange(len(volts)) * xinc

print(f'Captured {len(volts)} points, sample interval {xinc:.3e} s')

## Step 4: Plot and save

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(time_s * 1e3, volts)
plt.xlabel('Time (ms)')
plt.ylabel('Voltage (V)')
plt.title('Rigol DS1000Z capture')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

np.savetxt('capture.csv', np.column_stack([time_s, volts]), delimiter=',', header='time_s,volts', comments='')

## Cleanup

In [ ]:
scope.close()
rm.close()